In [ ]:
###Iniciando o spark

In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1791649099190_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
from datetime import datetime

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [62]:
dt_proc = datetime.now().strftime('%Y%m%d%H%M%S')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [54]:
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' >>>'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Início do processamento da camada trusted de faturas

In [55]:
df_fatura = spark.read.option('header','true').csv('s3://data-lake-elt-303004020922-us-east-1-an/01_raw/tb_fatura/')


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Para que serve:  
#Depois disso, você pode executar consultas SQL diretamente sobre esse DataFrame usando spark.sql()

In [56]:
df_fatura.createOrReplaceTempView("df_fatura")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [57]:
qtd_registros = df_fatura.count()
print(log(),f'Quantidade de Registros: {qtd_registros}')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2026-10-10 18:32:19 >>> Quantidade de Registros: 15553

In [58]:
df_fatura.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- id_cliente: string (nullable = true)
 |-- id_fatura: string (nullable = true)
 |-- dt_emissao_fatura: string (nullable = true)
 |-- dt_vencimento_fatura: string (nullable = true)
 |-- valor_fatura: string (nullable = true)

In [ ]:
#Definindo o schema para transformar os tipos e salvar a camada trusted

In [66]:
df_fatura_formated=spark.sql(f"""
          select
            cast(id_cliente as bigint) as id_cliente,
            cast(id_fatura as bigint) as id_fatura,
            "{dt_proc}" as dt_proc,
            substring(replace(dt_emissao_fatura,'-',''),1,6) as ref,
            to_date(dt_emissao_fatura) as dt_emissao_fatura,
            to_date(dt_vencimento_fatura) as dt_vencimento_fatura,
            cast(valor_fatura as decimal(14,2)) as valor_fatura
          from df_fatura            
          """)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
+----------+---------+--------------+------+-----------------+--------------------+------------+
|id_cliente|id_fatura|       dt_proc|   ref|dt_emissao_fatura|dt_vencimento_fatura|valor_fatura|
+----------+---------+--------------+------+-----------------+--------------------+------------+
|         1|        1|20261010183728|202301|       2023-01-01|          2023-01-06|     1671.07|
|         1|        2|20261010183728|202302|       2023-02-01|          2023-02-06|     3784.01|
|         1|        3|20261010183728|202303|       2023-03-01|          2023-03-06|     3265.83|
|         1|        4|20261010183728|202304|       2023-04-01|          2023-04-06|     2443.49|
+----------+---------+--------------+------+-----------------+--------------------+------------+

In [ ]:
#novamente criamos uma tabela temporária para trabalhar com sql

In [69]:
df_fatura_formated.createOrReplaceTempView('df_fatura_formated')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# O DataFrame já tratado (df_fatura_formated) e grava os dados em formato Parquet dentro do bucket S3, 
#   adicionando novos registros sem apagar os antigos, e organizando-os em partições por ref.

#Benefícios dessa abordagem:

# O formato Parquet é otimizado para análise (Spark, Athena, Redshift Spectrum).
# O particionamento por ref acelera consultas que filtram por essa coluna.
# O modo append permite cargas incrementais sem perder histórico.

In [ ]:
df_fatura_formated.write.mode('append').partitionBy('ref').parquet('s3://data-lake-elt-303004020922-us-east-1-an/02_trusted/tb_0001_fatura')

In [ ]:
# Adicionando uma camada de controle de processamento

In [82]:
df_controle_fatura = spark.sql(f"""
           select 
                   'tb_0001_fatura' as tb_name,
                   {dt_proc} as dt_proc,
                   {qtd_registros} as qtd_registros
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [83]:
df_controle_fatura.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------+--------------+-------------+
|       tb_name|       dt_proc|qtd_registros|
+--------------+--------------+-------------+
|tb_0001_fatura|20261010183728|        15553|
+--------------+--------------+-------------+

In [ ]:
df_controle_fatura.write.mode("append").parquet('s3://data-lake-elt-303004020922-us-east-1-an/04_controle/trunsted')